In [ ]:
%%capture
!pip uninstall -y transformers
!pip install transformers==4.44.2 datasets==2.19.0

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found ex

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/AncientRusProject_SpanCollator_V1"
TOKENIZER_DIR = f"{BASE_DIR}/ancient_rus_tokenizer"
DATA_FILE = f"{BASE_DIR}/ancient_rus_ready_for_bert.txt"
MODEL_DIR = f"{BASE_DIR}/mini_bert_ancient_rus"

In [ ]:
import torch
import random
import math
import numpy as np
from datasets import load_dataset
from transformers import (
    BertConfig, BertForMaskedLM, BertTokenizerFast,
    Trainer, TrainingArguments
)

In [ ]:
dataset = load_dataset("text", data_files={"train": DATA_FILE})
tokenizer = BertTokenizerFast.from_pretrained(TOKENIZER_DIR)

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
split_dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=False)

In [ ]:
tokenized_datasets = split_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/465075 [00:00<?, ? examples/s]

Map:   0%|          | 0/24478 [00:00<?, ? examples/s]

In [ ]:
def group_texts(examples):
    block_size = 256
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    return result

In [ ]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)

Map:   0%|          | 0/465075 [00:00<?, ? examples/s]

Map:   0%|          | 0/24478 [00:00<?, ? examples/s]

In [ ]:
config = BertConfig(
    vocab_size=len(tokenizer),
    hidden_size=512,
    num_hidden_layers=6,
    num_attention_heads=8,
    intermediate_size=2048,
    max_position_embeddings=512,
    pad_token_id=tokenizer.pad_token_id,
)

In [ ]:
model = BertForMaskedLM(config)
print(f"Bert model parameters: {model.num_parameters():,}")

🧠 Параметры модели: 34,833,202 (Оптимизировано для быстрого обучения)


In [ ]:
class PhysicalDegradationCollator:
    """
    Simulates real damage to historical documents:
    1. Broken edges (Edge Masking)
    2. Worn holes (Span Masking)
    3. Erased parts of the words (Random Subword Masking)
    """
    def __init__(self, tokenizer, mlm_prob=0.15, max_span=3, edge_prob=0.1):
        self.tokenizer = tokenizer
        self.mlm_prob = mlm_prob
        self.max_span = max_span
        self.edge_prob = edge_prob

    def __call__(self, features):
        input_ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
        attention_mask = torch.tensor([f["attention_mask"] for f in features], dtype=torch.long)
        labels = input_ids.clone()

        batch_size, seq_len = input_ids.shape
        probability_matrix = torch.full(labels.shape, self.mlm_prob)

        # Special tokens protection
        special_tokens_mask = [
            self.tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
        ]
        special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)

        # Basic random mask
        masked_indices = torch.bernoulli(probability_matrix).bool()
        final_mask = masked_indices.clone()



        for i in range(batch_size):
            # 1. Edge Masking
            if random.random() < self.edge_prob:
                edge_len = random.randint(2, 5)
                is_start = random.choice([True, False])

                # Looking for boundaries, ignoring <s> and </s> и and the context tags
                valid_indices = (~special_tokens_mask[i]).nonzero(as_tuple=True)[0]
                if len(valid_indices) > edge_len:
                    if is_start:
                        start_idx = valid_indices[0]
                        final_mask[i, start_idx : start_idx + edge_len] = True
                    else:
                        end_idx = valid_indices[-1]
                        final_mask[i, end_idx - edge_len + 1 : end_idx + 1] = True

            # 2. Span Masking
            for j in range(seq_len):
                if masked_indices[i, j]:
                    span_len = random.randint(1, self.max_span)
                    end_idx = min(j + span_len, seq_len)
                    if not special_tokens_mask[i, j:end_idx].any():
                        final_mask[i, j:end_idx] = True

        labels[~final_mask] = -100

        # Standard 80% [MASK], 10% random, 10% original
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & final_mask
        input_ids[indices_replaced] = self.tokenizer.mask_token_id

        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & final_mask & ~indices_replaced
        random_words = torch.randint(len(self.tokenizer), labels.shape, dtype=torch.long)
        input_ids[indices_random] = random_words[indices_random]

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
data_collator = PhysicalDegradationCollator(tokenizer=tokenizer, mlm_prob=0.12, max_span=3, edge_prob=0.15)

In [ ]:
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    # Оставляем только топ-5 предсказаний, чтобы не перегружать оперативную память (RAM)
    top_k_logits = torch.topk(logits, k=5, dim=-1).indices
    return top_k_logits

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    mask = labels != -100
    labels = labels[mask]
    preds = preds[mask]

    return {
        "top1_accuracy": np.mean(preds[:, 0] == labels),
        "top3_accuracy": np.mean(np.any(preds[:, :3] == labels[:, None], axis=1)),
        "top5_accuracy": np.mean(np.any(preds[:, :5] == labels[:, None], axis=1)),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    overwrite_output_dir=True,
    num_train_epochs=15,
    per_device_train_batch_size=64,
    gradient_accumulation_steps=2,
    evaluation_strategy="steps",
    eval_steps=400,
    save_steps=400,
    save_total_limit=2,
    logging_steps=100,
    prediction_loss_only=False,
    fp16=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=1000,
    weight_decay=0.01,
    report_to="none"
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],

    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Top1 Accuracy,Top3 Accuracy,Top5 Accuracy
400,7.466300,7.504405,0.129388,0.184902,0.210291
800,7.392100,7.340801,0.138475,0.194471,0.219862
1200,6.761500,6.633702,0.172317,0.230355,0.256682
1600,6.250000,6.154231,0.188625,0.252541,0.281951
2000,5.866300,5.744744,0.205428,0.276858,0.312477
2400,5.420800,5.288718,0.228874,0.314152,0.356998
2800,5.072900,4.937914,0.258762,0.354823,0.401506
3200,4.795700,4.661877,0.284359,0.391783,0.440214
3600,4.570500,4.444129,0.313993,0.423074,0.470898
4000,4.406700,4.293180,0.331797,0.444127,0.492199



🔮 --- ПРОВЕРКА НА ШАГЕ 400 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и [MASK] и ст҃го дх҃а  ->  ['и', '.', ',']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  ['.', 'и', ',']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['.', 'а', ',']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['.', 'и', ',']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['.', ',', 'и']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити [MASK]  ->  ['.', 'а', ',']
----------------------------------------------


🔮 --- ПРОВЕРКА НА ШАГЕ 400 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и [MASK] и ст҃го дх҃а  ->  ['и', '.', ',']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  ['.', 'и', ',']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['.', 'а', ',']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['.', 'и', ',']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['.', ',', 'и']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити [MASK]  ->  ['.', 

Step,Training Loss,Validation Loss,Top1 Accuracy,Top3 Accuracy,Top5 Accuracy
400,7.466300,7.504405,0.129388,0.184902,0.210291
800,7.392100,7.340801,0.138475,0.194471,0.219862
1200,6.761500,6.633702,0.172317,0.230355,0.256682
1600,6.250000,6.154231,0.188625,0.252541,0.281951
2000,5.866300,5.744744,0.205428,0.276858,0.312477
2400,5.420800,5.288718,0.228874,0.314152,0.356998
2800,5.072900,4.937914,0.258762,0.354823,0.401506
3200,4.795700,4.661877,0.284359,0.391783,0.440214
3600,4.570500,4.444129,0.313993,0.423074,0.470898
4000,4.406700,4.293180,0.331797,0.444127,0.492199



🔮 --- ПРОВЕРКА НА ШАГЕ 4400 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и [MASK] и ст҃го дх҃а  ->  ['сн҃а', 'оц҃а', 'ст҃го']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  [':', 'смену', 'фоносу']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['.', 'имати', 'судити']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['и', 'на', 'в']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['молодец', '!', 'конь']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити [MASK]  ->  ['.', ';', ':']
----------------------------------------------


🔮 --- ПРОВЕРКА НА ШАГЕ 4400 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и [MASK] и ст҃го дх҃а  ->  ['сн҃а', 'оц҃а', 'ст҃го']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  [':', 'смену', 'фоносу']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['.', 'имати', 'судити']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['и', 'на', 'в']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['молодец', 

There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


TrainOutput(global_step=5415, training_loss=5.486750203816844, metrics={'train_runtime': 4412.0777, 'train_samples_per_second': 156.997, 'train_steps_per_second': 1.227, 'total_flos': 2.04376981890816e+16, 'train_loss': 5.486750203816844, 'epoch': 15.0})

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

('/content/drive/MyDrive/AncientRusProject_SpanCollator_V1/mini_bert_ancient_rus/tokenizer_config.json',
 '/content/drive/MyDrive/AncientRusProject_SpanCollator_V1/mini_bert_ancient_rus/special_tokens_map.json',
 '/content/drive/MyDrive/AncientRusProject_SpanCollator_V1/mini_bert_ancient_rus/vocab.txt',
 '/content/drive/MyDrive/AncientRusProject_SpanCollator_V1/mini_bert_ancient_rus/added_tokens.json',
 '/content/drive/MyDrive/AncientRusProject_SpanCollator_V1/mini_bert_ancient_rus/tokenizer.json')

In [ ]:
print("\nFINAL RESULTS:")
eval_results = trainer.evaluate()
print(f"Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")
print(f"Top-1 Accuracy: {eval_results.get('eval_top1_accuracy', 0):.2%}")
print(f"Top-3 Accuracy: {eval_results.get('eval_top3_accuracy', 0):.2%}")
print(f"Top-5 Accuracy: {eval_results.get('eval_top5_accuracy', 0):.2%}")

📊 Результаты после 15 эпох:
Финишый Loss: 4.1090
Perplexity: 60.89
⚠️ Модели было сложно. Возможно, нужно больше данных или слоев.


In [ ]:
from transformers import pipeline

In [ ]:
fill_mask = pipeline(
    "fill-mask",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    device=0,
)

In [ ]:
test_cases = [
    # 1. Classic: checking cases and logic (Chronicles)
    {
        "desc": "📚 Летописи (на какую землю?)",
        "text": "[CTX_LIT] И пошелъ князь игорь на <mask> землю со своею дружиною.",
        "expected": "рускую / свою"
    },

    # 2. Sudebnic: testing knowledge of specific laws
    {
        "desc": "⚖️ Русская Правда (кого убили?)",
        "text": "[CTX_LEGAL] Аже кто оубиеть <mask> , то платити виру 40 гривенъ.",
        "expected": "мужь"
    },

    # 3. Daily: checking understanding of debts
    {
        "desc": "🏡 Грамоты (про что пишут?)",
        "text": "[CTX_DAILY] поклоне ѿ ꙩндреꙗ · к ѥва · и к микифору про <mask> ѡкупи ꙩсподине",
        "expected": "серебро"
    },

    # 4. Tear-off edge testing (Edge Masking)
    # ​The sentence has no beginning, but RoPE must understand that a bow is being sent to Vasily
    {
        "desc": "🧨 ТЕСТ ROPE: Оторванное начало",
        "text": "[CTX_DAILY] <mask> <mask> ко василью . а серебро ми отдай.",
        "expected": "поклонъ ѿ"
    },

    # 5. Tear-off edge testing (Edge Masking)
    {
        "desc": "🧨 ТЕСТ ROPE: Оторванный конец",
        "text": "[CTX_EPIC] Выезжал добрый <mask> из <mask> на <mask> <mask>",
        "expected": "молодец из города на добром коне"
    },

    # 6. [GAP] testing
    {
        "desc": "🧩 ТЕСТ [GAP]: Работа с нечитаемым текстом",
        "text": "[CTX_DAILY] [GAP] бь ѿ но [GAP] тию и св <mask> коуно",
        "expected": "Модель должна предложить варианты, игнорируя дыры [GAP]"
    },

    # 7. Church: plural check
    {
        "desc": "⛪️ Церковный (кому сказал?)",
        "text": "[CTX_CHURCH] И рече господь къ <mask> своимъ, глаголя...",
        "expected": "ученикомъ / людемъ"
    }
]

print("\n" + "=" * 60)
print(" BERT (Phisical Degradation collator)")
print("=" * 60)

for idx, case in enumerate(test_cases, 1):
    print(f"\n[{idx}/7] {case['desc']}")
    print(f"Text: {case['text']}")
    print(f"Expected (meaning): {case['expected']}")

    # If there are more than 5 masks
    mask_count = case['text'].count("<mask>")
    results = fill_mask(case['text'], top_k=3)

    # Output normalization
    if mask_count == 1:
        results = [results]

    for i, mask_res in enumerate(results):
        print(f"Mask {i+1}: ", end="")
        preds = []
        for res in mask_res:
            clean_word = res['token_str'].replace("Ġ", "").strip()
            score = res['score'] * 100
            preds.append(f"'{clean_word}' ({score:.1f}%)")
        print(" | ".join(preds))

🔍 Загрузка обученной модели для финального теста...

🏆 ФИНАЛЬНЫЙ ЭКЗАМЕН MINI-BERT (ВСЕ КАТЕГОРИИ)

🔹 ⛪️ [CTX_CHURCH] (Ожидаем: сына / отца / бога / духа)
Текст: [CTX_CHURCH] Во имя отца и [MASK] и святаго духа.
  1. сына         (Уверенность: 76.3%)
  2. отца         (Уверенность: 6.6%)
  3. духа         (Уверенность: 4.0%)

🔹 🏡 [CTX_DAILY] (Ожидаем: господину / брату / юрью)
Текст: [CTX_DAILY] Поклонъ ѿ бориса ко [MASK] съ бг҃омъ.
  1. василью      (Уверенность: 11.4%)
  2. мнѣ          (Уверенность: 7.1%)
  3. гн҃у         (Уверенность: 4.9%)

🔹 ⚖️ [CTX_LEGAL] (Ожидаем: винити / судити / имати / дати)
Текст: [CTX_LEGAL] Аже оубиеть моужь мужа, то мьстити брату, а посулов не [MASK] .
  1. имати        (Уверенность: 51.9%)
  2. надобѣ       (Уверенность: 7.7%)
  3. просити      (Уверенность: 5.4%)

🔹 📚 [CTX_LIT] (Ожидаем: словесы / дѣлы)
Текст: [CTX_LIT] Не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть.
  1. ,            (Уверенность: 13.4%)
  2. и            (Уверенн